# Realistic RAG Pipeline with RudriQ

This notebook demonstrates RudriQ on an enterprise-realistic workload: 5 source documents, ~200 pandas operations, 43 LLM calls, all traced into a unified graph with cross-domain causal links.

**What you'll see:**
- Standard pandas + OpenAI code with no RudriQ-specific instrumentation
- RudriQ automatically captures every operation
- The unified trace renders as an audit report ready for compliance review

OpenAI HTTP layer is mocked via `httpx.MockTransport`, so this notebook runs offline and deterministically. Drop in a real `OPENAI_API_KEY` and the pipeline runs identically against live OpenAI.

## 1. Activate RudriQ

One import wires AutoLineage's pandas hooks, the OpenAI SDK monkey-patches, and the lid-propagation patches. We then initialize Traceloop (which sets up an OpenTelemetry SDK TracerProvider) and attach a `RudriQSpanProcessor` to it.

In [ ]:
import os
# Set the env var BEFORE rudriq.auto runs - rudriq.auto calls
# Traceloop.init at import time, which bails before swapping the
# global TracerProvider when no API key is present.
os.environ.setdefault('TRACELOOP_API_KEY', 'test-key-not-real')

# Suppress the OTLP HTTP exporter's 401 noise: with a fake API key,
# every flush logs an Unauthorized error. The exporter still runs
# (RudriQ's local SpanProcessor receives spans normally); we just
# silence the cloud-shipping log for a cleaner offline demo.
import logging
for _name in (
    'opentelemetry.exporter.otlp.proto.http.trace_exporter',
    'opentelemetry.exporter.otlp.proto.http.metric_exporter',
    'opentelemetry.exporter.otlp.proto.grpc.trace_exporter',
    'opentelemetry.exporter.otlp.proto.grpc.metric_exporter',
):
    logging.getLogger(_name).setLevel(logging.CRITICAL)

import rudriq.auto

from traceloop.sdk import Traceloop
Traceloop.init(app_name='rudriq-realistic-demo', disable_batch=True)

from opentelemetry import trace
from rudriq.processors import RudriQSpanProcessor

provider = trace.get_tracer_provider()
rudriq_proc = RudriQSpanProcessor()
provider.add_span_processor(rudriq_proc)

print('RudriQ initialized.')
print(f'Run ID: {rudriq_proc.run_id}')

## 2. Mock OpenAI client

The cell below configures a mock HTTP transport that returns canned embeddings and chat responses. This makes the notebook runnable without an API key. In production, you would skip this cell entirely and let the real OpenAI client run — RudriQ instruments it identically.

In [ ]:
import json, time
import httpx
from openai import OpenAI

def _mock_handler(request):
    url = str(request.url)
    body = json.loads(request.content) if request.content else {}
    if 'embeddings' in url:
        inputs = body.get('input', [])
        if isinstance(inputs, str):
            inputs = [inputs]
        n = len(inputs) or 1
        return httpx.Response(200, json={
            'object': 'list',
            'data': [
                {'object': 'embedding', 'index': i,
                 'embedding': [0.01 * (i + j) for j in range(8)]}
                for i in range(n)
            ],
            'model': body.get('model', 'text-embedding-3-small'),
            'usage': {'prompt_tokens': n * 5, 'total_tokens': n * 5},
        })
    if 'chat/completions' in url:
        return httpx.Response(200, json={
            'id': 'chatcmpl-mock',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': body.get('model', 'gpt-4'),
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant',
                            'content': 'Mock response based on retrieved context.'},
                'finish_reason': 'stop',
            }],
            'usage': {'prompt_tokens': 100, 'completion_tokens': 20,
                      'total_tokens': 120},
        })
    return httpx.Response(404, json={'error': 'unknown endpoint'})

client = OpenAI(
    api_key='sk-test-fake',
    http_client=httpx.Client(transport=httpx.MockTransport(_mock_handler)),
)
print('Mock OpenAI client ready.')

## 3. Read source documents

Five CSV files representing different document collections. Each `pd.read_csv` is automatically captured by AutoLineage as a data lineage operation.

In [ ]:
import tempfile
from pathlib import Path
import pandas as pd
import numpy as np

tmp_dir = Path(tempfile.gettempdir()) / 'rudriq_demo_data'
tmp_dir.mkdir(exist_ok=True)

for i in range(5):
    p = tmp_dir / f'docs_{i}.csv'
    if not p.exists():
        pd.DataFrame({
            'doc_id': [f'd{i}_{j}' for j in range(50)],
            'text': [f'document {i}.{j} about topic {j % 10}' for j in range(50)],
            'lang': ['en' if j % 3 != 0 else 'es' for j in range(50)],
            'score': np.random.RandomState(i).rand(50),
            'category': [f'cat_{j % 5}' for j in range(50)],
        }).to_csv(p, index=False)

dataframes = [pd.read_csv(tmp_dir / f'docs_{i}.csv') for i in range(5)]
print(f'Read {len(dataframes)} CSVs, '
      f'{sum(len(d) for d in dataframes)} total rows.')

## 4. Pandas transformations

For each DataFrame: filter to English documents, deduplicate, sort by score, take the top 30. Then concatenate and group. Every operation is captured with input/output shape and timing.

In [ ]:
transformed = []
for df in dataframes:
    df_en = df[df['lang'] == 'en']
    df_dedup = df_en.drop_duplicates(subset=['text'])
    df_sorted = df_dedup.sort_values('score', ascending=False)
    df_top = df_sorted.head(30)
    transformed.append(df_top)

combined = pd.concat(transformed, ignore_index=True)
grouped = combined.groupby('category').agg({'score': 'mean'}).reset_index()
final_docs = combined.sort_values('score', ascending=False).reset_index(drop=True)

print(f'Combined: {len(combined)} rows from 5 inputs')
print(f'Grouped:  {len(grouped)} categories')
print(f'Final:    {len(final_docs)} rows ready for embedding')

## 5. Batched embeddings

Embed in batches of 50. Each `client.embeddings.create` call produces an LLM operation in the trace, and RudriQ's linker correlates the input back to upstream pandas operations via element-identity matching.

In [ ]:
texts = final_docs['text'].tolist()
batch_size = 50
all_embeddings = []

for batch_idx in range(0, len(texts), batch_size):
    batch = texts[batch_idx:batch_idx + batch_size]
    response = client.embeddings.create(
        model='text-embedding-3-small',
        input=batch,
    )
    all_embeddings.extend(e.embedding for e in response.data)

embeddings_array = np.array(all_embeddings)
print(f'{len(all_embeddings)} embeddings in '
      f'{(len(texts) + batch_size - 1) // batch_size} batches')
print(f'Index shape: {embeddings_array.shape}')

## 6. Query workload

For each of 20 queries: embed the query, retrieve the top-5 docs by cosine similarity, format a prompt, call chat completions. This is the typical RAG production loop.

In [ ]:
queries = [f'Tell me about topic {i}' for i in range(20)]

for q_idx, query in enumerate(queries):
    q_resp = client.embeddings.create(model='text-embedding-3-small', input=query)
    q_vec = np.array(q_resp.data[0].embedding)

    norms = (np.linalg.norm(embeddings_array, axis=1)
             * np.linalg.norm(q_vec) + 1e-9)
    scores = embeddings_array @ q_vec / norms
    top5 = scores.argsort()[-5:][::-1]
    top_docs = [texts[i] for i in top5]

    messages = [
        {'role': 'system', 'content': 'Answer based on the provided context.'},
        {'role': 'user', 'content': f"Context:\n{chr(10).join(top_docs)}\n\nQuestion: {query}"},
    ]
    client.chat.completions.create(model='gpt-4', messages=messages)

print(f'Ran {len(queries)} queries (each: 1 embed + 1 chat = 2 LLM calls).')

## 7. What was captured

Standard LLM observability tools show you the LLM calls — 43 of them in this run. RudriQ shows the full unified graph: every data operation that contributed, every cross-domain link, the complete provenance chain.

In [ ]:
from collections import Counter
from rudriq.storage import get_default_storage

graph = get_default_storage().load_run(rudriq_proc.run_id)

print(f'Total operations: {len(graph.nodes)}')
data_count = sum(1 for n in graph.nodes if n.kind.value.startswith('data_'))
llm_count = sum(1 for n in graph.nodes if n.kind.value.startswith('llm_'))
print(f'  Data operations: {data_count}')
print(f'  LLM operations:  {llm_count}')

print(f'\nTotal edges: {len(graph.edges)}')
for kind, count in sorted(Counter(e.kind.value for e in graph.edges).items()):
    print(f'  {kind}: {count}')

llm_ids = {n.node_id for n in graph.nodes if n.kind.value.startswith('llm_')}
linked = {e.child_id for e in graph.edges
          if e.kind.value == 'lineage' and e.child_id in llm_ids}
print(f'\nLLM calls linked to upstream data: {len(linked)} / {len(llm_ids)}')

### Why some LLM calls aren't linked yet

In this demo, the **batched embedding calls** link cleanly via object identity — each batch is a slice of a list whose elements were registered by RudriQ when `df['text'].tolist()` ran. The **query embeddings** receive fresh strings (not from a tracked DataFrame), and the **chat completions** receive freshly-built `messages` dicts whose content is a prompt-formatted string. Neither matches by object identity in principle.

Object-identity is one of three linker strategies. Content-hash and substring-with-provenance matching for prompt-formatted inputs are tracked in [BACKLOG.md](../BACKLOG.md) as the v0.0.8+ retrieval-aware linker. The mechanism on display here is the foundation; broader coverage is the next deliverable.

## 8. Walk one LLM call's full upstream chain

Here's the lineage chain for one of the linked calls — every operation that contributed to its input, in causal order back to the source CSV.

In [ ]:
from rudriq.export.audit import _compute_lineage_chains

chains = _compute_lineage_chains(graph)
linked_chains = [c for c in chains if c['chain_length'] > 0]

if not linked_chains:
    print('No linked LLM calls in this run — see BACKLOG for retrieval-aware linker.')
else:
    chain = max(linked_chains, key=lambda c: c['chain_length'])
    print(f"Lineage chain for {chain['llm_library']}.{chain['llm_operation']}:")
    print(f"  LLM call started:      {chain['llm_started_at']}")
    print(f"  Upstream chain length: {chain['chain_length']} steps")
    print()
    for step in chain['chain'][:15]:
        indent = '    ' * (step['depth'] - 1)
        print(f"{indent}depth {step['depth']}: {step['library']}.{step['operation']}")
    if chain['chain_length'] > 15:
        print(f"    ... and {chain['chain_length'] - 15} more steps")

## 9. The audit report

This is the artifact a compliance officer reviews. It contains the summary, every LLM call's full upstream lineage chain, and a complete operations appendix. Below shows the first part inline; the full report is saved to `audit_report.md`.

In [ ]:
from rudriq.export.audit import export_audit_markdown

audit_md = export_audit_markdown(rudriq_proc.run_id)

with open('audit_report.md', 'w', encoding='utf-8') as f:
    f.write(audit_md)

print(audit_md[:2500])
print('\n... [truncated for inline display] ...')
print(f'\nFull report ({len(audit_md)} chars) saved to audit_report.md')

## What this enables

A self-contained audit artifact you can hand to:

- **Your compliance team** when they ask "what data was used to generate this AI response?"
- **Your auditor** for technical documentation per EU AI Act Annex IV
- **Your AI insurance underwriter** for evidence of data provenance
- **Your debugging session** when an AI response is flagged

All running locally. No outbound network calls in core. Air-gapped capable.

For more, see the [RudriQ README](https://github.com/kishanraj41/rudriq).